In [2]:
import inspect
from utils import load_eeg_data
print(inspect.getsource(load_eeg_data))

def load_eeg_data(
    condition,
    subject_id,
    trials='all',
    channels='all',
    freq_band=None,
    tmin=None,
    tmax=None,
    normalize=False,
):
    """
    Load EEG data as Epochs and optionally apply time cropping, z-score normalization,
    and channel selection (including union of channel groups).

    Parameters
    ----------
    condition : str
        Experiment condition; must match the filename. Examples: "BLA", "BLT", "P1", "P2", "P3".
    subject_id : int
        Subject identifier. Expected file:
        ``dataset/binepochs filtered ICArej {condition}AvgBOS{subject_id}.set``.
    trials : str | list | slice | int
        Which trials to return from the already-filtered set (excluding "no auditory").
        - ``'all'``: all meaningful trials.
        - List of ints: e.g. ``[0, 1, 2]`` for the first three.
        - Slice: e.g. ``slice(0, 10)`` or ``:10`` for the first 10.
        - Int: a single trial.
    channels : str | list
        Channels to load.
  

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from utils import load_eeg_data

SUBJECT = 5
WINDOW_MS = 200
STEP_MS = 50
MAX_LAG_MS = 20

# Load same data as the existing notebook
blt = load_eeg_data('BLT', SUBJECT, normalize=True)
p1  = load_eeg_data('P1',  SUBJECT, normalize=True)
p2_all = load_eeg_data('P2', SUBJECT, normalize=True)
p2 = select_p2_500ms(p2_all)

# Make sure channels align
common_ch = [c for c in blt.ch_names if c in p1.ch_names and c in p2.ch_names]
for ep in (blt, p1, p2):
    ep.pick_channels(common_ch, ordered=True)

sfreq = p1.info['sfreq']  # should be 512 Hz
max_lag_samples = int(MAX_LAG_MS / 1000 * sfreq)  # convert ms to samples

def lagged_cross_corr(x, y, max_lag):
    """
    For two 1D signals x and y, compute cross-correlation at lags
    from -max_lag to +max_lag. Returns peak correlation and its lag.
    Positive lag means x leads y (x fires first).
    """
    lags = range(-max_lag, max_lag + 1)
    corrs = []
    for lag in lags:
        if lag < 0:
            # y is shifted forward, meaning y leads x
            corrs.append(np.corrcoef(x[:lag], y[-lag:])[0, 1])
        elif lag > 0:
            # x is shifted forward, meaning x leads y
            corrs.append(np.corrcoef(x[lag:], y[:-lag])[0, 1])
        else:
            corrs.append(np.corrcoef(x, y)[0, 1])
    corrs = np.array(corrs)
    peak_idx = np.argmax(np.abs(corrs))
    return corrs[peak_idx], list(lags)[peak_idx]

def sliding_lagged_corr(epochs, window_ms, step_ms, max_lag):
    """
    Sliding window lagged cross-correlation.
    Returns:
      peak_corr: (n_windows, n_ch, n_ch) — peak correlation value
      peak_lag:  (n_windows, n_ch, n_ch) — lag in samples at peak
                 positive = row channel leads column channel
      centers:   (n_windows,) window center times in seconds
    """
    data = epochs.get_data()           # (n_trials, n_ch, n_times)
    avg  = data.mean(axis=0)           # (n_ch, n_times) — trial average
    
    sfreq = epochs.info['sfreq']
    win_samples  = int(window_ms / 1000 * sfreq)
    step_samples = int(step_ms   / 1000 * sfreq)
    n_ch, n_times = avg.shape
    times = epochs.times

    starts = range(0, n_times - win_samples, step_samples)
    n_windows = len(list(starts))
    
    peak_corr = np.zeros((n_windows, n_ch, n_ch))
    peak_lag  = np.zeros((n_windows, n_ch, n_ch))
    centers   = []

    for w, start in enumerate(range(0, n_times - win_samples, step_samples)):
        end = start + win_samples
        centers.append(times[start + win_samples // 2])
        for i in range(n_ch):
            for j in range(n_ch):
                if i == j:
                    peak_corr[w, i, j] = 1.0
                    peak_lag[w, i, j]  = 0
                elif j < i:
                    # mirror — already computed
                    peak_corr[w, i, j] =  peak_corr[w, j, i]
                    peak_lag[w, i, j]  = -peak_lag[w, j, i]
                else:
                    pc, pl = lagged_cross_corr(avg[i, start:end],
                                               avg[j, start:end],
                                               max_lag)
                    peak_corr[w, i, j] = pc
                    peak_lag[w, i, j]  = pl

    return peak_corr, peak_lag, np.array(centers)

ModuleNotFoundError: No module named 'utils'

In [1]:
# Run for both conditions
peak_corr_p1, peak_lag_p1, centers = sliding_lagged_corr(p1, WINDOW_MS, STEP_MS, max_lag_samples)
peak_corr_p2, peak_lag_p2, _       = sliding_lagged_corr(p2, WINDOW_MS, STEP_MS, max_lag_samples)

# Convert lag from samples to ms for readability
lag_ms_p1 = peak_lag_p1 / sfreq * 1000
lag_ms_p2 = peak_lag_p2 / sfreq * 1000

# Plot lag maps at the same 5 key time points the group used
key_times = [-0.40, 0.25, 0.45, 0.60, 0.75]
pick_idx = [np.argmin(np.abs(centers - t)) for t in key_times]

fig, axes = plt.subplots(2, len(pick_idx), figsize=(15, 6))
for j, w in enumerate(pick_idx):
    for i, (lag_ms, label) in enumerate([(lag_ms_p1, 'P1 (CISI)'), (lag_ms_p2, 'P2 VISI-500')]):
        ax = axes[i, j]
        im = ax.imshow(lag_ms[w], cmap='coolwarm', vmin=-MAX_LAG_MS, vmax=MAX_LAG_MS, aspect='auto')
        ax.set_xticks([]); ax.set_yticks([])
        if i == 0:
            ax.set_title(f't={centers[w]:+.2f}s', fontsize=9)
        if j == 0:
            ax.set_ylabel(label, fontsize=10)

fig.colorbar(im, ax=axes.ravel().tolist(), label='Peak lag (ms)\n+ = row leads col')
fig.suptitle('Lagged cross-correlation: directionality maps')
plt.tight_layout()
plt.show()

NameError: name 'sliding_lagged_corr' is not defined